# Brain Tumor Detector — Improved Training Pipeline

**Model:** MobileNetV2 (ImageNet pretrained, fine-tuned)  
**Key improvements over v1 (EfficientNetB0):**
- Brain-region contour cropping removes background noise  
- Correct MobileNetV2 preprocessing: pixel range → [-1, 1]  
- Class-weighted loss to handle "no_tumor" imbalance  
- Three-phase training with LR scheduling + EarlyStopping  
- Expected accuracy: **90–96%** (was 34–35%)  
- Model size: ~9 MB, 3.4 M params (vs 15 MB / 4 M EfficientNetB0)

In [ ]:
import os
import json
import numpy as np
import cv2
import imutils
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Configuration

In [ ]:
TRAIN_DIR = '../data/train'
VAL_DIR   = '../data/val'
MODEL_DIR = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
SEED        = 42
NUM_CLASSES = 4
CLASS_NAMES = ['glioma', 'meningioma', 'no_tumor', 'pituitary']

PHASE1_EPOCHS = 20
PHASE1_LR     = 1e-3

PHASE2_EPOCHS = 40
PHASE2_LR     = 1e-4
FINE_TUNE_AT  = 100   # unfreeze MobileNetV2 layers from index 100 onward (of 155)

BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'tumor_classifier.h5')
print('Config ready.')

## 2. Brain-Region Cropping
Adapted from MohamedAliHabib/Brain-Tumor-Detection.  
Removes dark skull/background so the model focuses on brain tissue only.

In [ ]:
def crop_brain_contour(image_np):
    """Detect brain region via contour detection and crop tightly around it."""
    try:
        gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)
        _, thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)
        thresh = cv2.erode(thresh, None, iterations=2)
        thresh = cv2.dilate(thresh, None, iterations=2)
        cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = imutils.grab_contours(cnts)
        if not cnts:
            return image_np
        c = max(cnts, key=cv2.contourArea)
        ext_left  = tuple(c[c[:, :, 0].argmin()][0])
        ext_right = tuple(c[c[:, :, 0].argmax()][0])
        ext_top   = tuple(c[c[:, :, 1].argmin()][0])
        ext_bot   = tuple(c[c[:, :, 1].argmax()][0])
        cropped = image_np[ext_top[1]:ext_bot[1], ext_left[0]:ext_right[0]]
        if cropped.shape[0] < 10 or cropped.shape[1] < 10:
            return image_np
        return cropped
    except Exception:
        return image_np

print('crop_brain_contour defined.')

## 3. Data Loading with Brain Cropping + MobileNetV2 Preprocessing

In [ ]:
def load_dataset(directory, class_names, img_size=(224, 224)):
    """
    Load images, apply brain cropping, resize, then apply
    MobileNetV2 preprocess_input (scales to [-1, 1]).
    """
    X, y = [], []
    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(directory, class_name)
        if not os.path.isdir(class_dir):
            print(f'  WARNING: {class_dir} not found, skipping.')
            continue
        files = [f for f in os.listdir(class_dir)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f'  {class_name}: {len(files)} images')
        for fname in files:
            img = cv2.imread(os.path.join(class_dir, fname))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = crop_brain_contour(img)           # ← key step
            img = cv2.resize(img, img_size, interpolation=cv2.INTER_CUBIC)
            img = preprocess_input(img.astype(np.float32))  # → [-1, 1]
            X.append(img)
            y.append(class_idx)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


print('Loading training data...')
X_train, y_train = load_dataset(TRAIN_DIR, CLASS_NAMES, IMG_SIZE)
print('\nLoading validation data...')
X_val,   y_val   = load_dataset(VAL_DIR,   CLASS_NAMES, IMG_SIZE)

print(f'\nX_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val  : {X_val.shape}    y_val  : {y_val.shape}')

## 4. Class Weights (handles imbalance)

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes.tolist(), weights.tolist()))

print('Class weights (higher = under-represented):')
for idx, name in enumerate(CLASS_NAMES):
    print(f'  {name:15s}: {class_weight_dict[idx]:.3f}')

## 5. Augmentation Generators (training only)

In [ ]:
train_datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    brightness_range=[0.85, 1.15],
    shear_range=5,
)

val_datagen = ImageDataGenerator()  # no augmentation for validation

train_gen = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED)
val_gen   = val_datagen.flow(X_val,   y_val,   batch_size=BATCH_SIZE, shuffle=False)

print('Generators created.')

## 6. Model — MobileNetV2 + Custom Head

| | MobileNetV2 | EfficientNetB0 (old) |
|---|---|---|
| Params | 3.4 M | 5.3 M |
| Top-1 ImageNet | 71.8% | 77.1% |
| Inference (CPU) | Fast | Moderate |
| Architecture | Depthwise separable (designed for edge) | Compound scaling |

In [ ]:
def build_model(num_classes, input_shape=(224, 224, 3), fine_tune_at=None):
    base = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')

    if fine_tune_at is None:
        base.trainable = False
    else:
        base.trainable = True
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return Model(inputs=base.input, outputs=outputs), base


model, base_model = build_model(NUM_CLASSES)

trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
total     = sum(np.prod(v.shape) for v in model.variables)
print(f'Trainable params (Phase 1): {trainable:,}')
print(f'Total params              : {total:,}')
model.summary()

## 7. Phase 1 — Train Head (frozen backbone)

In [ ]:
cb_phase1 = [
    callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor='val_accuracy',
        save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=4, min_lr=1e-6, verbose=1
    ),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history1 = model.fit(
    train_gen,
    epochs=PHASE1_EPOCHS,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=cb_phase1,
    verbose=1
)
print(f'\nPhase 1 best val_accuracy: {max(history1.history["val_accuracy"]):.4f}')

## 8. Phase 2 — Fine-tune Top Layers

In [ ]:
model, base_model = build_model(NUM_CLASSES, fine_tune_at=FINE_TUNE_AT)
model.load_weights(BEST_MODEL_PATH)

trainable_ft = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Phase 2 trainable params: {trainable_ft:,}')

cb_phase2 = [
    callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor='val_accuracy',
        save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=12,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=5, min_lr=1e-7, verbose=1
    ),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_gen,
    epochs=PHASE2_EPOCHS,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=cb_phase2,
    verbose=1
)
print(f'\nPhase 2 best val_accuracy: {max(history2.history["val_accuracy"]):.4f}')

## 9. Training Curves

In [ ]:
acc1,  vacc1 = history1.history['accuracy'], history1.history['val_accuracy']
acc2,  vacc2 = history2.history['accuracy'], history2.history['val_accuracy']
loss1, vloss1 = history1.history['loss'],    history1.history['val_loss']
loss2, vloss2 = history2.history['loss'],    history2.history['val_loss']

ep = range(1, len(acc1) + len(acc2) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep, acc1  + acc2,  label='Train')
axes[0].plot(ep, vacc1 + vacc2, label='Val')
axes[0].axvline(len(acc1), color='gray', linestyle='--', label='Phase 2 start')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(ep, loss1  + loss2,  label='Train')
axes[1].plot(ep, vloss1 + vloss2, label='Val')
axes[1].axvline(len(loss1), color='gray', linestyle='--', label='Phase 2 start')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('../models/training_curves.png', dpi=120)
plt.show()

## 10. Evaluation & Confusion Matrix

In [ ]:
model.load_weights(BEST_MODEL_PATH)

val_loss, val_acc = model.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)
print(f'Validation Accuracy : {val_acc*100:.2f}%')
print(f'Validation Loss     : {val_loss:.4f}')

y_pred = np.argmax(model.predict(X_val, batch_size=BATCH_SIZE), axis=1)
print('\nClassification Report:')
print(classification_report(y_val, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.ylabel('True label'); plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig('../models/confusion_matrix.png', dpi=120)
plt.show()

## 11. Save Label Map

In [ ]:
label_map = {name: idx for idx, name in enumerate(CLASS_NAMES)}
with open(os.path.join(MODEL_DIR, 'label_map.json'), 'w') as f:
    json.dump(label_map, f, indent=2)

print('\n' + '='*50)
print('  TRAINING COMPLETE')
print('='*50)
print(f'  Model         : MobileNetV2')
print(f'  Val Accuracy  : {val_acc*100:.2f}%')
print(f'  Model path    : {BEST_MODEL_PATH}')
print('='*50)